# Assignment 5 — Part B2: Loan Approval Classification

**Dataset:** `clean_loan_dataset.csv` (produced by Part B1)  
**Goal:** Train and evaluate three classifiers — Logistic Regression, Random Forest, and a third algorithm (Support Vector Machine) — on the loan approval prediction task.  
**Positive class:** `Approved = 1`

## Step 1 — Load Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv('clean_loan_dataset.csv')

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
print()
print(df.head())

Dataset loaded successfully.
Shape: (100, 9)

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445243               8.274536  
1                 0         1     -0.912753               0.153994  
2                 0         1      0.280078              -0.126321  
3                 0         1     -0.315204              -0.669033  
4                 0         1     -0.284704              -0.284975  


## Step 2 — Prepare Features & Target

In [2]:
# Target: Approved (1 = Approved, 0 = Rejected)
# Features: all columns except Approved

X = df.drop(columns=['Approved'])
y = df['Approved']

print('Features (X):', list(X.columns))
print('Target  (y): Approved')
print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print()
print('Target distribution:')
print(y.value_counts())

Features (X): ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'HasCollateral', 'PreviousDefaults', 'DebtToIncome', 'IncomePerYearEmployed']
Target  (y): Approved
X shape: (100, 8)  |  y shape: (100,)

Target distribution:
Approved
1    66
0    34
Name: count, dtype: int64


## Step 3 — Split Data

In [3]:
# 80% training, 20% testing | stratified split preserves class balance | random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training set:  {X_train.shape[0]} rows')
print(f'Testing set:   {X_test.shape[0]} rows')
print()
print('Train class distribution:')
print(y_train.value_counts())
print()
print('Test class distribution:')
print(y_test.value_counts())

Training set:  80 rows
Testing set:   20 rows

Train class distribution:
Approved
1    53
0    27
Name: count, dtype: int64

Test class distribution:
Approved
1    13
0     7
Name: count, dtype: int64


## Step 4 — Train Models (Reproduce Lesson)

In [4]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print('Logistic Regression trained.')

# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print('Random Forest trained.')

Logistic Regression trained.


Random Forest trained.


## Step 5 — Research & Train a Third Classifier

### Third Classifier: Support Vector Machine (SVC)

I chose **Support Vector Machine (SVC)** as my third classifier for the following reasons:

1. **How it works:** SVM finds the optimal **hyperplane** that maximally separates the two classes in the feature space. It maximizes the margin between the nearest data points of each class (the "support vectors"). Using the **RBF kernel** (Radial Basis Function), SVM can capture non-linear decision boundaries — making it well-suited for the complex, multi-feature loan dataset.

2. **Why it fits loan approval:** After scaling with `RobustScaler` in Part B1, all features are on a comparable scale, which is a key requirement for SVM to work effectively. Loan approval decisions often involve non-linear interactions between features (e.g., high income can offset a lower credit score), which the RBF kernel can model.

3. **One advantage:** Strong generalization even on small datasets — SVM only uses the support vectors (boundary points) to define the decision boundary, making it robust to noise in the interior of each class.

4. **One limitation:** Computationally expensive on very large datasets and sensitive to the choice of kernel and hyperparameters (`C`, `gamma`).

**Source:** Scikit-learn documentation — *SVC (Support Vector Classification)*: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

In [5]:
# --- Support Vector Machine (SVC) ---
# kernel='rbf' for non-linear boundaries, C=1.0 default regularization, probability=True to enable predict_proba
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm_model.fit(X_train, y_train)
print('Support Vector Machine (SVC) trained.')

Support Vector Machine (SVC) trained.


## Step 6 — Evaluate Performance

In [6]:
def evaluate_model(name, model, X_test, y_test):
    """Print Accuracy, Precision, Recall, F1-Score, and Confusion Matrix for a trained model."""
    y_pred = model.predict(X_test)
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec  = recall_score(y_test, y_pred, pos_label=1)
    f1   = f1_score(y_test, y_pred, pos_label=1)
    cm   = confusion_matrix(y_test, y_pred)
    
    print(f'{name} Performance:')
    print(f'  Accuracy : {acc:.3f}')
    print(f'  Precision: {prec:.3f}  (positive = Approved=1)')
    print(f'  Recall   : {rec:.3f}  (positive = Approved=1)')
    print(f'  F1-Score : {f1:.3f}  (positive = Approved=1)')
    print()
    print(f'  Confusion Matrix:')
    print(f'                    Predicted Approved  Predicted Rejected')
    print(f'  Actually Approved      {cm[1][1]}                  {cm[1][0]}')
    print(f'  Actually Rejected      {cm[0][1]}                  {cm[0][0]}')
    print()
    print('-' * 60)
    print()
    return y_pred

In [7]:
lr_preds  = evaluate_model('Logistic Regression',          lr_model,  X_test, y_test)
rf_preds  = evaluate_model('Random Forest',                rf_model,  X_test, y_test)
svm_preds = evaluate_model('Support Vector Machine (SVC)', svm_model, X_test, y_test)

Logistic Regression Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

  Confusion Matrix:
                    Predicted Approved  Predicted Rejected
  Actually Approved      11                  2
  Actually Rejected      4                  3

------------------------------------------------------------

Random Forest Performance:
  Accuracy : 0.650
  Precision: 0.714  (positive = Approved=1)
  Recall   : 0.769  (positive = Approved=1)
  F1-Score : 0.741  (positive = Approved=1)

  Confusion Matrix:
                    Predicted Approved  Predicted Rejected
  Actually Approved      10                  3
  Actually Rejected      4                  3

------------------------------------------------------------

Support Vector Machine (SVC) Performance:
  Accuracy : 0.650
  Precision: 0.688  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.75

## Step 7 — Sanity Check

Pick one row from the test set and compare all three model predictions against the actual label.

In [8]:
# Pick index 5 from the test set for inspection
sanity_index = 5
sample = X_test.iloc[[sanity_index]]
actual_label = y_test.iloc[sanity_index]

lr_pred  = lr_model.predict(sample)[0]
rf_pred  = rf_model.predict(sample)[0]
svm_pred = svm_model.predict(sample)[0]

label_map = {1: 'Approved', 0: 'Rejected'}

print('=== SANITY CHECK — Single Row Prediction ===')
print(f'Row index in test set: {sanity_index}')
print()
print(f'  Actual Label                  : {label_map[actual_label]} ({actual_label})')
print(f'  Logistic Regression Prediction: {label_map[lr_pred]} ({lr_pred})')
print(f'  Random Forest Prediction       : {label_map[rf_pred]} ({rf_pred})')
print(f'  SVM Prediction                 : {label_map[svm_pred]} ({svm_pred})')
print()
print('Sample feature values:')
print(sample.to_string())

=== SANITY CHECK — Single Row Prediction ===
Row index in test set: 5

  Actual Label                  : Approved (1)
  Logistic Regression Prediction: Rejected (0)
  Random Forest Prediction       : Rejected (0)
  SVM Prediction                 : Rejected (0)

Sample feature values:
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  PreviousDefaults  DebtToIncome  IncomePerYearEmployed
80 -0.61568     0.841424         0.952991         0.0              0                 0       0.63818              -0.628362
